In [1]:
#### %cd /kaggle/working
!rm -rf anuraset
!git clone https://github.com/soundclim/anuraset.git
%cd anuraset
!pip install -q pyyaml pandas numpy scikit-learn librosa tqdm matplotlib torchmetrics
print("repo ready:", __import__("os").path.exists("baseline/configs/exp_resnet18.yaml"))

Cloning into 'anuraset'...
remote: Enumerating objects: 155, done.
remote: Counting objects: 100% (155/155), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 155 (delta 85), reused 99 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (155/155), 4.76 MiB | 13.10 MiB/s, done.
Resolving deltas: 100% (85/85), done.
/kaggle/working/anuraset
repo ready: True


In [2]:
import os, glob, pandas as pd
base  = "/kaggle/input/datasets/mismaresenka/anuraset-preprocessed/anuraset"
AUDIO = os.path.join(base, "audio")
stem2rel = {os.path.splitext(os.path.basename(p))[0]: os.path.relpath(p, AUDIO)
            for p in glob.glob(os.path.join(AUDIO, "**", "*.wav"), recursive=True)}
df = pd.read_csv(os.path.join(base, "metadata.csv"))
def key(r): return f"{r.fname}_{int(float(r.min_t))}_{int(float(r.max_t))}"
frac = df.apply(key, axis=1).isin(stem2rel).mean()
print(f"match: {frac:.1%}")
assert frac > 0.95, "STOP — filename match failed, tell Claude"
df[df.columns[0]] = df.apply(lambda r: stem2rel.get(key(r)), axis=1)
assert df[df.columns[0]].notna().all(), "STOP — some rows unmatched"
root = "/kaggle/working/anuraset_data"; os.makedirs(root, exist_ok=True)
df.to_csv(os.path.join(root, "metadata.csv"), index=False)
if not os.path.exists(os.path.join(root, "audio")): os.symlink(AUDIO, os.path.join(root, "audio"))
print("✓ data fixed —", len(df), "rows resolve")

match: 100.0%
✓ data fixed — 93378 rows resolve


In [3]:
import pandas as pd, numpy as np, ast

# weights
df = pd.read_csv("/kaggle/working/anuraset_data/metadata.csv")
labels = df[df["subset"]=="train"].iloc[:, 8:].values
pos = labels.sum(axis=0); neg = len(labels) - pos
pos_weight = np.clip(neg / np.clip(pos, 1, None), 1, 50)
np.save("/kaggle/working/pos_weight.npy", pos_weight.astype("float32"))
print("weights ready — min:", pos_weight.min().round(2), "max:", pos_weight.max().round(2))

tp = "/kaggle/working/anuraset/baseline/train.py"
src = open(tp).read()
I = " " * 8

# 1) if a previous (broken or good) patch is present, strip it back to the original line
import re
src = re.sub(
    r' *import numpy as _np, torch as _torch\n'
    r' *_pw = _torch\.tensor\(_np\.load\([^\n]*\)\n'
    r' *loss_fn = nn\.BCEWithLogitsLoss\(pos_weight=_pw\)',
    I + 'loss_fn = nn.BCEWithLogitsLoss()',
    src
)

# 2) now apply the correctly-indented patch to the clean line
inject = (I + 'import numpy as _np, torch as _torch\n'
          + I + '_pw = _torch.tensor(_np.load("/kaggle/working/pos_weight.npy")).to(device)\n'
          + I + 'loss_fn = nn.BCEWithLogitsLoss(pos_weight=_pw)')
src = src.replace(I + "loss_fn = nn.BCEWithLogitsLoss()", inject)
open(tp, "w").write(src)

print("patched:", "pos_weight=_pw" in open(tp).read())
try:
    ast.parse(open(tp).read()); print("train.py is valid Python ✓")
except Exception as e:
    print("STILL BROKEN:", e)

weights ready — min: 3.69 max: 50.0
patched: True
train.py is valid Python ✓


In [4]:
import yaml, shutil, os
# wipe the leftover smoke-test model so training starts clean
shutil.rmtree("/kaggle/working/anuraset/baseline/model_states", ignore_errors=True)
cfgp = "/kaggle/working/anuraset/baseline/configs/exp_resnet18.yaml"
cfg = yaml.safe_load(open(cfgp))
cfg.update({"data_root": "/kaggle/working/anuraset_data", "num_epochs": 10, "batch_size": 16})
yaml.safe_dump(cfg, open(cfgp, "w"))
print("config set for REAL run: 10 epochs")

config set for REAL run: 10 epochs


In [5]:
import subprocess, sys, os
os.chdir("/kaggle/working/anuraset")
print("training for real — this takes a while...\n" + "="*45)
r = subprocess.run([sys.executable, "baseline/train.py", "--config",
                    "baseline/configs/exp_resnet18.yaml"], capture_output=True, text=True)
print(r.stdout[-3000:]); print("STDERR:", r.stderr[-1500:]); print("="*45, "EXIT:", r.returncode)

training for real — this takes a while...
Using config "baseline/configs/exp_resnet18.yaml"
Using device cuda
There are 62191 samples in the training set.
There are 31187 samples in the test set.
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
Starting new model
Starting training
Epoch: 1: Loss val: 0.5003 ; F1-score macro val: 0.1834 - Epoch time: 754.2s; Total time: 754.2s - 10%
Epoch: 2: Loss val: 0.4297 ; F1-score macro val: 0.2012 - Epoch time: 690.2s; Total time: 1444.5s - 20%
Epoch: 3: Loss val: 0.4547 ; F1-score macro val: 0.2033 - Epoch time: 695.9s; Total time: 2140.5s - 30%
Epoch: 4: Loss val: 0.6395 ; F1-score macro val: 0.2043 - Epoch time: 688.2s; Total time: 2828.8s - 40%
Epoch: 5: Loss val: 0.5380 ; F1-score macro val: 0.1985 - Epoch time: 677.6s; Total time: 3506.5s - 50%
Epoch: 6: Loss val: 0.6150 ; F1-score macro val: 0.2133 - Epoch time: 703.6s; Total time: 4210.1s - 60%
Epoch: 7: L